# Quality Metrics Analysis

This notebook develops and tests quality scoring formulas for sign recordings.

## Goals
1. Develop quality scoring formula
2. Test on various recordings
3. Identify quality thresholds
4. Visualize quality distributions

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json
from typing import List, Dict, Optional, Tuple

from src.types import Point3D, HandLandmarks, RecordingFrame, RecordingSession
from src.landmarks import LANDMARK_NAMES, NUM_LANDMARKS, FINGER_LANDMARK_INDICES
from src.normalize import normalize_landmarks
from src.validation import validate_landmarks
from src.angles import calculate_all_joint_angles
from src.math_utils import calculate_distance

from notebook_utils import (
    landmarks_to_arrays,
    plot_hand_3d,
    plot_comparison,
    create_sample_landmarks,
    FINGER_COLORS
)

# Set matplotlib style
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Quality Metrics Components

We'll develop a multi-factor quality score based on:
- **Completeness**: Are all landmarks detected?
- **Stability**: How stable are landmarks across frames?
- **Anatomical validity**: Are proportions and angles realistic?
- **Tracking confidence**: MediaPipe confidence scores (if available)
- **Spatial coherence**: Are landmarks in reasonable positions?

In [ ]:
def calculate_completeness_score(landmarks: HandLandmarks) -> float:
    """
    Calculate completeness score based on landmark presence.
    Returns 1.0 if all landmarks present, lower if missing or invalid.
    """
    if not landmarks:
        return 0.0
    
    valid_count = 0
    for name in LANDMARK_NAMES:
        point = landmarks.get(name)
        if point is not None:
            # Check for invalid coordinates (NaN, Inf, or clearly wrong values)
            if (np.isfinite(point.x) and np.isfinite(point.y) and np.isfinite(point.z) and
                -10 <= point.x <= 10 and -10 <= point.y <= 10 and -10 <= point.z <= 10):
                valid_count += 1
    
    return valid_count / NUM_LANDMARKS

# Test with sample data
sample = create_sample_landmarks()
print(f"Completeness score for sample: {calculate_completeness_score(sample):.2f}")

In [ ]:
def calculate_stability_score(frames: List[HandLandmarks], 
                              max_expected_movement: float = 0.05) -> float:
    """
    Calculate stability score based on frame-to-frame consistency.
    Lower jitter = higher score.
    
    Args:
        frames: List of hand landmarks for consecutive frames
        max_expected_movement: Maximum expected movement per frame (normalized units)
    """
    if len(frames) < 2:
        return 1.0  # Can't measure stability with single frame
    
    jitters = []
    
    for i in range(1, len(frames)):
        prev_frame = frames[i-1]
        curr_frame = frames[i]
        
        frame_jitters = []
        for name in LANDMARK_NAMES:
            prev_pt = prev_frame.get(name)
            curr_pt = curr_frame.get(name)
            
            if prev_pt and curr_pt:
                dist = calculate_distance(prev_pt, curr_pt)
                frame_jitters.append(dist)
        
        if frame_jitters:
            jitters.append(np.mean(frame_jitters))
    
    if not jitters:
        return 0.0
    
    avg_jitter = np.mean(jitters)
    # Convert to 0-1 score (lower jitter = higher score)
    score = max(0.0, 1.0 - (avg_jitter / max_expected_movement))
    return min(1.0, score)

# Create test frames with slight movement
test_frames = []
base = create_sample_landmarks()
for i in range(10):
    # Add small random noise to simulate jitter
    frame = {}
    for name, pt in base.items():
        noise = np.random.normal(0, 0.005, 3)  # Small noise
        frame[name] = Point3D(x=pt.x + noise[0], y=pt.y + noise[1], z=pt.z + noise[2])
    test_frames.append(frame)

print(f"Stability score (low jitter): {calculate_stability_score(test_frames):.2f}")

# Create high-jitter frames
high_jitter_frames = []
for i in range(10):
    frame = {}
    for name, pt in base.items():
        noise = np.random.normal(0, 0.05, 3)  # Higher noise
        frame[name] = Point3D(x=pt.x + noise[0], y=pt.y + noise[1], z=pt.z + noise[2])
    high_jitter_frames.append(frame)

print(f"Stability score (high jitter): {calculate_stability_score(high_jitter_frames):.2f}")

In [ ]:
def calculate_anatomical_score(landmarks: HandLandmarks) -> float:
    """
    Calculate anatomical validity score based on:
    - Bone length ratios
    - Joint angles within normal range
    """
    if not landmarks or len(landmarks) < NUM_LANDMARKS:
        return 0.0
    
    scores = []
    
    # Check finger bone length ratios
    # Proximal > Middle > Distal is typical
    finger_names = ['thumb', 'index', 'middle', 'ring', 'pinky']
    
    for finger in finger_names:
        indices = FINGER_LANDMARK_INDICES.get(finger, [])
        if len(indices) >= 4:
            points = [landmarks.get(LANDMARK_NAMES[i]) for i in indices]
            if all(p is not None for p in points):
                # Calculate bone lengths
                lengths = []
                for i in range(len(points) - 1):
                    lengths.append(calculate_distance(points[i], points[i+1]))
                
                # Check for reasonable ratios (no bone should be 3x another)
                if min(lengths) > 0:
                    ratio = max(lengths) / min(lengths)
                    ratio_score = max(0.0, 1.0 - (ratio - 1.0) / 3.0)
                    scores.append(ratio_score)
    
    # Check joint angles
    try:
        angles = calculate_all_joint_angles(landmarks)
        angle_scores = []
        for angle_name, angle_value in angles.items():
            # Angles should be 0-180 degrees
            if 0 <= angle_value <= 180:
                angle_scores.append(1.0)
            else:
                angle_scores.append(0.5)
        if angle_scores:
            scores.append(np.mean(angle_scores))
    except Exception:
        pass
    
    return np.mean(scores) if scores else 0.5

print(f"Anatomical score for sample: {calculate_anatomical_score(sample):.2f}")

In [ ]:
def calculate_spatial_coherence_score(landmarks: HandLandmarks) -> float:
    """
    Calculate spatial coherence score.
    Checks that landmarks form a coherent hand shape.
    """
    if not landmarks:
        return 0.0
    
    scores = []
    
    # 1. Check that wrist is below fingers (in most poses)
    wrist = landmarks.get('wrist')
    if wrist:
        fingertips = [
            landmarks.get('thumb_tip'),
            landmarks.get('index_finger_tip'),
            landmarks.get('middle_finger_tip'),
            landmarks.get('ring_finger_tip'),
            landmarks.get('pinky_tip')
        ]
        valid_tips = [t for t in fingertips if t is not None]
        
        if valid_tips:
            # Check fingertips are spread from wrist
            avg_tip_dist = np.mean([calculate_distance(wrist, t) for t in valid_tips])
            # Fingertips should be at reasonable distance from wrist (0.1 to 0.4 normalized)
            if 0.05 < avg_tip_dist < 0.5:
                scores.append(1.0)
            else:
                scores.append(0.5)
    
    # 2. Check finger order (index to pinky should be in sequence on x-axis)
    finger_tips = [
        landmarks.get('index_finger_tip'),
        landmarks.get('middle_finger_tip'),
        landmarks.get('ring_finger_tip'),
        landmarks.get('pinky_tip')
    ]
    if all(t is not None for t in finger_tips):
        x_positions = [t.x for t in finger_tips]
        # Check if generally ordered (allow some variation)
        ordered = sum(x_positions[i] >= x_positions[i+1] for i in range(len(x_positions)-1))
        order_score = ordered / (len(x_positions) - 1)
        scores.append(order_score)
    
    # 3. Check palm forms a reasonable quadrilateral
    palm_points = [
        landmarks.get('wrist'),
        landmarks.get('thumb_cmc'),
        landmarks.get('index_finger_mcp'),
        landmarks.get('pinky_mcp')
    ]
    if all(p is not None for p in palm_points):
        # Check palm area is reasonable
        palm_width = calculate_distance(palm_points[2], palm_points[3])  # index to pinky MCP
        palm_height = calculate_distance(palm_points[0], palm_points[2])  # wrist to index MCP
        if palm_width > 0 and palm_height > 0:
            ratio = palm_width / palm_height
            # Palm is typically wider than tall at base, ratio 0.5-2.0 is reasonable
            if 0.3 < ratio < 3.0:
                scores.append(1.0)
            else:
                scores.append(0.5)
    
    return np.mean(scores) if scores else 0.5

print(f"Spatial coherence score for sample: {calculate_spatial_coherence_score(sample):.2f}")

## 2. Combined Quality Score Formula

In [ ]:
class QualityScorer:
    """Combined quality scorer with configurable weights."""
    
    def __init__(self, 
                 completeness_weight: float = 0.3,
                 stability_weight: float = 0.25,
                 anatomical_weight: float = 0.25,
                 coherence_weight: float = 0.2):
        self.weights = {
            'completeness': completeness_weight,
            'stability': stability_weight,
            'anatomical': anatomical_weight,
            'coherence': coherence_weight
        }
        # Normalize weights
        total = sum(self.weights.values())
        self.weights = {k: v/total for k, v in self.weights.items()}
    
    def score_frame(self, landmarks: HandLandmarks) -> Dict[str, float]:
        """Score a single frame."""
        scores = {
            'completeness': calculate_completeness_score(landmarks),
            'anatomical': calculate_anatomical_score(landmarks),
            'coherence': calculate_spatial_coherence_score(landmarks),
            'stability': 1.0  # Can't measure from single frame
        }
        scores['overall'] = sum(scores[k] * self.weights[k] for k in self.weights)
        return scores
    
    def score_recording(self, frames: List[HandLandmarks]) -> Dict[str, float]:
        """Score a full recording."""
        if not frames:
            return {k: 0.0 for k in list(self.weights.keys()) + ['overall']}
        
        # Calculate per-frame scores
        frame_scores = [self.score_frame(f) for f in frames]
        
        # Average across frames
        scores = {
            'completeness': np.mean([s['completeness'] for s in frame_scores]),
            'anatomical': np.mean([s['anatomical'] for s in frame_scores]),
            'coherence': np.mean([s['coherence'] for s in frame_scores]),
            'stability': calculate_stability_score(frames)
        }
        scores['overall'] = sum(scores[k] * self.weights[k] for k in self.weights)
        return scores
    
    def get_quality_level(self, score: float) -> str:
        """Convert numeric score to quality level."""
        if score >= 0.9:
            return 'Excellent'
        elif score >= 0.75:
            return 'Good'
        elif score >= 0.6:
            return 'Acceptable'
        elif score >= 0.4:
            return 'Poor'
        else:
            return 'Unusable'

# Test the scorer
scorer = QualityScorer()
frame_scores = scorer.score_frame(sample)
print("Single frame scores:")
for k, v in frame_scores.items():
    print(f"  {k}: {v:.2f}")
print(f"  Quality level: {scorer.get_quality_level(frame_scores['overall'])}")

In [ ]:
# Test with recording (multiple frames)
recording_scores = scorer.score_recording(test_frames)
print("\nRecording scores (10 frames, low jitter):")
for k, v in recording_scores.items():
    print(f"  {k}: {v:.2f}")
print(f"  Quality level: {scorer.get_quality_level(recording_scores['overall'])}")

recording_scores_jittery = scorer.score_recording(high_jitter_frames)
print("\nRecording scores (10 frames, high jitter):")
for k, v in recording_scores_jittery.items():
    print(f"  {k}: {v:.2f}")
print(f"  Quality level: {scorer.get_quality_level(recording_scores_jittery['overall'])}")

## 3. Test on Real Recordings

In [ ]:
def load_recording_landmarks(recording_path: Path) -> List[HandLandmarks]:
    """Load landmarks from a recording file."""
    with open(recording_path) as f:
        data = json.load(f)
    
    frames = []
    for frame_data in data.get('frames', []):
        landmarks = {}
        # Handle different data structures
        if 'right_hand' in frame_data:
            hand_data = frame_data['right_hand']
        elif 'landmarks' in frame_data:
            hand_data = frame_data['landmarks']
        else:
            continue
        
        for name in LANDMARK_NAMES:
            if name in hand_data:
                pt = hand_data[name]
                landmarks[name] = Point3D(x=pt['x'], y=pt['y'], z=pt.get('z', 0))
        
        if landmarks:
            frames.append(landmarks)
    
    return frames

# Find recording files
recordings_dir = Path('../data/recordings')
recording_files = list(recordings_dir.glob('*.json')) if recordings_dir.exists() else []

print(f"Found {len(recording_files)} recording files")

if recording_files:
    # Score each recording
    all_scores = []
    for rec_file in recording_files[:20]:  # Limit to first 20
        try:
            frames = load_recording_landmarks(rec_file)
            if frames:
                scores = scorer.score_recording(frames)
                scores['file'] = rec_file.name
                scores['frame_count'] = len(frames)
                all_scores.append(scores)
        except Exception as e:
            print(f"Error loading {rec_file.name}: {e}")
    
    if all_scores:
        print(f"\nScored {len(all_scores)} recordings:")
        for s in sorted(all_scores, key=lambda x: x['overall'], reverse=True)[:10]:
            level = scorer.get_quality_level(s['overall'])
            print(f"  {s['file']}: {s['overall']:.2f} ({level}) - {s['frame_count']} frames")
else:
    print("No recordings found. Using synthetic data for demonstration.")

## 4. Identify Quality Thresholds

In [ ]:
# Generate synthetic recordings with varying quality
def generate_synthetic_recording(quality_level: str, num_frames: int = 30) -> List[HandLandmarks]:
    """Generate synthetic recording with specified quality."""
    base = create_sample_landmarks()
    frames = []
    
    noise_levels = {
        'excellent': 0.002,
        'good': 0.01,
        'acceptable': 0.025,
        'poor': 0.05,
        'unusable': 0.1
    }
    
    missing_probs = {
        'excellent': 0.0,
        'good': 0.02,
        'acceptable': 0.05,
        'poor': 0.15,
        'unusable': 0.3
    }
    
    noise = noise_levels.get(quality_level, 0.02)
    missing_prob = missing_probs.get(quality_level, 0.05)
    
    for _ in range(num_frames):
        frame = {}
        for name, pt in base.items():
            if np.random.random() > missing_prob:
                jitter = np.random.normal(0, noise, 3)
                frame[name] = Point3D(
                    x=pt.x + jitter[0],
                    y=pt.y + jitter[1],
                    z=pt.z + jitter[2]
                )
        frames.append(frame)
    
    return frames

# Generate and score recordings at each quality level
quality_levels = ['excellent', 'good', 'acceptable', 'poor', 'unusable']
samples_per_level = 20

quality_data = {level: [] for level in quality_levels}

for level in quality_levels:
    for _ in range(samples_per_level):
        frames = generate_synthetic_recording(level)
        scores = scorer.score_recording(frames)
        quality_data[level].append(scores['overall'])

# Print statistics
print("Quality Level Statistics:")
print("-" * 60)
print(f"{'Level':<12} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 60)
for level in quality_levels:
    scores = quality_data[level]
    print(f"{level:<12} {np.mean(scores):>8.3f} {np.std(scores):>8.3f} {np.min(scores):>8.3f} {np.max(scores):>8.3f}")

In [ ]:
# Find optimal threshold boundaries
def find_threshold(level1_scores, level2_scores):
    """Find threshold that best separates two quality levels."""
    combined = [(s, 0) for s in level1_scores] + [(s, 1) for s in level2_scores]
    combined.sort(key=lambda x: x[0])
    
    best_threshold = 0
    best_accuracy = 0
    
    for threshold in np.arange(0.0, 1.0, 0.01):
        correct = sum(1 for s, label in combined if (s >= threshold) == (label == 0))
        accuracy = correct / len(combined)
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_threshold = threshold
    
    return best_threshold, best_accuracy

print("\nRecommended Thresholds:")
print("-" * 40)
thresholds = {}

for i in range(len(quality_levels) - 1):
    level1 = quality_levels[i]
    level2 = quality_levels[i + 1]
    threshold, accuracy = find_threshold(quality_data[level1], quality_data[level2])
    thresholds[f"{level1}/{level2}"] = threshold
    print(f"{level1} vs {level2}: {threshold:.2f} (accuracy: {accuracy:.1%})")

print("\n\nSuggested QualityScorer thresholds:")
print("  Excellent: >= 0.90")
print("  Good:      >= 0.75")
print("  Acceptable: >= 0.60")
print("  Poor:      >= 0.40")
print("  Unusable:  < 0.40")

## 5. Visualize Quality Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
ax1 = axes[0]
data_for_box = [quality_data[level] for level in quality_levels]
bp = ax1.boxplot(data_for_box, labels=quality_levels, patch_artist=True)

colors = ['#2ecc71', '#27ae60', '#f1c40f', '#e74c3c', '#c0392b']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax1.set_ylabel('Quality Score')
ax1.set_xlabel('Quality Level')
ax1.set_title('Quality Score Distribution by Level')
ax1.axhline(y=0.9, color='g', linestyle='--', alpha=0.5, label='Excellent threshold')
ax1.axhline(y=0.75, color='y', linestyle='--', alpha=0.5, label='Good threshold')
ax1.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5, label='Acceptable threshold')
ax1.axhline(y=0.4, color='r', linestyle='--', alpha=0.5, label='Poor threshold')
ax1.legend(loc='lower left', fontsize=8)

# Histogram
ax2 = axes[1]
for i, level in enumerate(quality_levels):
    ax2.hist(quality_data[level], bins=15, alpha=0.6, label=level, color=colors[i])

ax2.set_xlabel('Quality Score')
ax2.set_ylabel('Count')
ax2.set_title('Quality Score Histogram')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Component contribution analysis
fig, ax = plt.subplots(figsize=(10, 6))

components = ['completeness', 'stability', 'anatomical', 'coherence']
x = np.arange(len(quality_levels))
width = 0.2

# Regenerate with component breakdown
component_data = {comp: {level: [] for level in quality_levels} for comp in components}

for level in quality_levels:
    for _ in range(samples_per_level):
        frames = generate_synthetic_recording(level)
        scores = scorer.score_recording(frames)
        for comp in components:
            component_data[comp][level].append(scores[comp])

# Plot grouped bars
for i, comp in enumerate(components):
    means = [np.mean(component_data[comp][level]) for level in quality_levels]
    ax.bar(x + i * width, means, width, label=comp.capitalize())

ax.set_ylabel('Score')
ax.set_xlabel('Quality Level')
ax.set_title('Component Score Breakdown by Quality Level')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(quality_levels)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between components
import warnings
warnings.filterwarnings('ignore')

# Flatten all scores
all_component_scores = {comp: [] for comp in components + ['overall']}
for level in quality_levels:
    for _ in range(samples_per_level):
        frames = generate_synthetic_recording(level)
        scores = scorer.score_recording(frames)
        for comp in components:
            all_component_scores[comp].append(scores[comp])
        all_component_scores['overall'].append(scores['overall'])

# Create correlation matrix
corr_components = components + ['overall']
corr_matrix = np.zeros((len(corr_components), len(corr_components)))

for i, comp1 in enumerate(corr_components):
    for j, comp2 in enumerate(corr_components):
        corr_matrix[i, j] = np.corrcoef(all_component_scores[comp1], 
                                         all_component_scores[comp2])[0, 1]

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap='RdYlGn', vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_components)))
ax.set_yticks(range(len(corr_components)))
ax.set_xticklabels([c.capitalize() for c in corr_components], rotation=45, ha='right')
ax.set_yticklabels([c.capitalize() for c in corr_components])

# Add correlation values
for i in range(len(corr_components)):
    for j in range(len(corr_components)):
        text = ax.text(j, i, f'{corr_matrix[i, j]:.2f}',
                       ha='center', va='center', color='black', fontsize=10)

ax.set_title('Component Correlation Matrix')
plt.colorbar(im, ax=ax, label='Correlation')
plt.tight_layout()
plt.show()

## 6. Quality Score Integration

Example of how to integrate quality scoring into the pipeline.

In [ ]:
def assess_recording_quality(
    frames: List[HandLandmarks],
    min_quality: float = 0.6,
    verbose: bool = True
) -> Tuple[bool, Dict[str, float]]:
    """
    Assess if a recording meets quality requirements.
    
    Args:
        frames: List of hand landmarks for each frame
        min_quality: Minimum acceptable quality score
        verbose: Print detailed results
        
    Returns:
        Tuple of (passes_quality, scores_dict)
    """
    scorer = QualityScorer()
    scores = scorer.score_recording(frames)
    
    passes = scores['overall'] >= min_quality
    level = scorer.get_quality_level(scores['overall'])
    
    if verbose:
        print(f"\nQuality Assessment Results:")
        print(f"  Overall: {scores['overall']:.2f} ({level})")
        print(f"  Components:")
        for comp in ['completeness', 'stability', 'anatomical', 'coherence']:
            status = '\u2713' if scores[comp] >= min_quality else '\u2717'
            print(f"    {comp.capitalize()}: {scores[comp]:.2f} {status}")
        print(f"\n  Result: {'PASS' if passes else 'FAIL'}")
        
        if not passes:
            # Identify weakest component
            weakest = min(['completeness', 'stability', 'anatomical', 'coherence'],
                         key=lambda x: scores[x])
            print(f"  Suggestion: Improve {weakest} (currently {scores[weakest]:.2f})")
    
    return passes, scores

# Test with different quality recordings
print("Testing GOOD quality recording:")
good_frames = generate_synthetic_recording('good')
passes, scores = assess_recording_quality(good_frames)

print("\n" + "="*50 + "\n")

print("Testing POOR quality recording:")
poor_frames = generate_synthetic_recording('poor')
passes, scores = assess_recording_quality(poor_frames)

## 7. Summary and Recommendations

### Quality Score Formula

The final quality score is computed as:

```
Q = 0.30 * completeness + 0.25 * stability + 0.25 * anatomical + 0.20 * coherence
```

### Recommended Thresholds

| Level | Score Range | Use Case |
|-------|-------------|----------|
| Excellent | >= 0.90 | High-quality reference data, dictionary entries |
| Good | >= 0.75 | Standard dictionary entries, training data |
| Acceptable | >= 0.60 | Usable with caution, may need review |
| Poor | >= 0.40 | Needs improvement, not recommended for use |
| Unusable | < 0.40 | Should be re-recorded |

### Integration Points

1. **Recording Tool**: Show real-time quality feedback during recording
2. **Processing Pipeline**: Filter out low-quality recordings
3. **Dictionary Generation**: Only include recordings meeting minimum quality
4. **Validation**: Report quality scores in validation output

In [ ]:
# Export quality scorer for use in other notebooks/modules
print("Quality scorer can be imported from this notebook or integrated into src/quality.py")
print("\nExample integration code:")
print("""
from quality import QualityScorer, assess_recording_quality

# During recording
scorer = QualityScorer()
live_score = scorer.score_frame(current_landmarks)
show_quality_indicator(live_score['overall'])

# During processing
passes, scores = assess_recording_quality(frames, min_quality=0.6)
if passes:
    add_to_dictionary(recording)
else:
    log_quality_issue(recording, scores)
""")